GPU được kiểm tra sau khi cài dependencies pinned, bằng tiến trình Python mới ở cell smoke.
Chạy notebook từ đầu để mount Drive và cập nhật repo trước khi chạy experiment.


# 04D — Portable local / Colab setup and experiments

Default: **non-training smoke test**. Locally, use the current checkout and a pinned Python environment; set `EDGEAI_DATA_ROOT` before importing tools to use an external EdgeAI storage tree. Colab mounts Drive, clones or fast-forward pulls the GitHub repository, and installs its pinned runtime requirements. Code executes in the connected kernel's filesystem.

Upload the complete staged `EdgeAI` tree to `/content/drive/MyDrive/EdgeAI` and commit/push local code fixes before the first Colab run. Never run 00–03B to repair a missing cache. This notebook does not push code or upload data.


In [1]:
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
try:
    from google.colab import drive
except ImportError:
    drive = None
IN_COLAB = drive is not None

REPO_URL = "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git"
if IN_COLAB:
    drive.mount("/content/drive")
    assert REPO_URL.startswith("https://github.com/") and "@" not in REPO_URL
    repo_hint = os.environ.get("EDGEAI_REPO_ROOT", "/content/edge-ai")
    if PureWindowsPath(repo_hint).drive:
        raise ValueError("Colab needs a runtime checkout, not a Windows repository path")
    repo = Path(repo_hint).resolve()
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    else:
        origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
        assert origin == REPO_URL, "Existing checkout has a different origin"
        assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), "Preserve uncommitted runtime changes before pulling"
        subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    hint = Path(os.environ.get("EDGEAI_REPO_ROOT", Path.cwd())).resolve()
    repo = next((p for p in [hint, *hint.parents] if (p / "tools/project_paths.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout")
os.environ["EDGEAI_REPO_ROOT"] = str(repo)
os.chdir(repo)
sys.path.insert(0, str(repo / "tools"))
sys.dont_write_bytecode = True
print("Mode:", "COLAB" if IN_COLAB else "LOCAL")
print("Checkout:", repo)


Mounted at /content/drive
Mode: COLAB
Checkout: /content/edge-ai


In Colab, install the pinned cache-consumer and notebook dependencies. Locally, use the already configured environment without installing packages. Verification and smoke testing use fresh child Python processes. Configure private GitHub authentication interactively; never embed credentials here.


In [2]:
from colab_bootstrap import configure
paths = configure(install=IN_COLAB)


Repository: /content/edge-ai
Data: /content/drive/MyDrive/EdgeAI
Immutable baseline: /content/drive/MyDrive/EdgeAI/fixed/baseline/stage04
Future results: /content/drive/MyDrive/EdgeAI/experiments/<experiment_id>


Kiểm tra full frozen identities và baseline đã lưu; không train hoặc tạo lại dữ liệu.
Có thể bỏ cell này khi fixed inputs đã được audit PASS và không thay đổi.
GPU smoke/preflight ở cell tiếp theo vẫn cần chạy; không bỏ hash gate.


In [3]:
subprocess.run([sys.executable, "-B", "tools/verify_portability.py"], cwd=repo, check=True)


CompletedProcess(args=['/usr/bin/python3', '-B', 'tools/verify_portability.py'], returncode=0)

Colab yêu cầu CUDA. Smoke theo config exp010 kiểm tra đúng một DS-CNN/03A bằng input giả; local dùng CPU. Smoke không tạo optimizer hoặc train.


In [ ]:
device = "cuda" if IN_COLAB else "cpu"
subprocess.run([sys.executable, "-B", "tools/experiment_runner.py", "smoke", "--device", device, "--config", "configs/experiments/time_mask_v6.json"], cwd=repo, check=True)
print("Future outputs:", paths.EXPERIMENT_ROOT / "<experiment_id>")


## Lần tiếp theo: exp_010_time_mask — chỉ DS-CNN/03A

Đối chiếu exp007, chỉ thêm time masking khi tối ưu trên TRAIN: xác suất 0,5, một mask 1–8 frame trên toàn tần số, điền 0 sau TRAIN z-score. Không che dữ liệu khi TRAIN eval hoặc validation. Giữ LR=0,001, weight decay=0, CE exponent=0,75, seed42, batch64, max50, patience8, split/cache và quy tắc checkpoint như exp007. TEST tắt.

Chạy preview với `RUN_TRAINING=False`, kiểm tra protocol v6 và kế hoạch đúng một run. Sau smoke PASS mới bật True. Thư mục mới dưới Drive; tên đã có bị từ chối. Không cần chạy lại 00–03B hoặc 04A/04B. [Hướng dẫn exp010](../../docs/EXP_010_TIME_MASK.md).


In [ ]:
# CHỈNH ĐIỀU KHIỂN TẠI ĐÂY; mỗi lần train dùng tên chưa tồn tại.
EXPERIMENT_NAME = "exp_010_time_mask"
EXPERIMENT_DESCRIPTION = "Lần 10: DS-CNN/03A; chỉ thêm TRAIN time masking p=0.5, width 1-8 frame, zero sau z-score; đối chứng exp007, TEST tắt."
EXPERIMENT_CONFIG = "configs/experiments/time_mask_v6.json"
RUN_TRAINING = False

from cache_consumer import read_json
import experiment_runner
from experiment_runner import run_plan, training_module
config_preview = read_json(repo / EXPERIMENT_CONFIG)
config_preview.update(experiment_id=EXPERIMENT_NAME, description=EXPERIMENT_DESCRIPTION)
protocol = config_preview.get("protocol_version")
if protocol and protocol not in getattr(experiment_runner, "VERSIONED_PROTOCOLS", ()):
    raise RuntimeError(
        f"Loaded runner does not support {protocol}. Pull the latest repo, restart the kernel, "
        "then rerun setup and preview with RUN_TRAINING=False. "
        f"Loaded module: {experiment_runner.__file__}"
    )
plan = run_plan(config_preview)
engine = training_module(config_preview)
effective = (engine.effective_config(config_preview["training_overrides"], config_preview["evaluate_test"])
             if "protocol_version" in config_preview else engine.CONFIG)
experiment_path = paths.experiment(EXPERIMENT_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Kết quả:", experiment_path)
print("Reference:", config_preview.get("reference_experiment_id", "accepted baseline"))
print("Primary comparison in 04C:", "exp_007_power075 / 03A_ds_cnn_seed42")
print("Params thực thi:", effective)
if hasattr(engine, "class_weight_record"):
    from cache_consumer import CLASSES, COUNTS, TRAIN_COUNTS
    accepted_weights = {label: COUNTS["train"] / (4 * count)
                        for label, count in zip(CLASSES, TRAIN_COUNTS)}
    print("TRAIN class weights:", engine.class_weight_record(effective, accepted_weights))
print("Kế hoạch:", len(plan), "cấu hình, chạy lần lượt:", [p["run_id"] for p in plan])
if RUN_TRAINING:
    subprocess.run([
        sys.executable, "-u", "-B", "tools/experiment_runner.py", "train",
        "--config", EXPERIMENT_CONFIG, "--experiment-id", EXPERIMENT_NAME,
        "--description", EXPERIMENT_DESCRIPTION,
    ], cwd=repo, check=True)
else:
    print("Training disabled. Kiểm tra tên/config/params, rồi đổi RUN_TRAINING thành True khi muốn chạy.")


## Xem kết quả khi kế hoạch hoàn tất (exp010: 1/1 run)

Cell này chỉ đọc/xác thực kết quả; không train hoặc chạy lại TEST. Trong 04C, đặt cùng `EXPERIMENT_NAME` để xem history và confusion matrix của cả 8 cấu hình.


In [ ]:
if RUN_TRAINING:
    from experiment_review import compare_saved_results
    from IPython.display import display
    display(compare_saved_results(EXPERIMENT_NAME))
else:
    print("Để xem lần đã chạy, mở 04C và nhập EXPERIMENT_NAME tương ứng; không cần bật training.")
